# Phi-4 Original Only — Resume Training from Checkpoint + Evaluation

Run cells from top to bottom. This notebook copies the last checkpoint from Kaggle input/output data to `/kaggle/working`, resumes training, then runs inference/evaluation and saves CSV files.

In [2]:
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)

bitsandbytes version: 0.49.2


In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
True
Tesla T4


In [4]:
# ============================================================
# CELL 1 — Environment setup
# ============================================================
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment variables set. If CUDA was already initialized, restart session then run all.")

Environment variables set. If CUDA was already initialized, restart session then run all.


In [5]:
# ============================================================
# CELL 2 — Imports + config (NO sklearn)
# ============================================================
!pip install -q rapidfuzz
import os, json, random, warnings, re, time, gc, shutil, zipfile, inspect
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from tabulate import tabulate

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    TrainerCallback,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training,
)
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from rapidfuzz import fuzz as rfuzz

warnings.filterwarnings("ignore")
NOTEBOOK_START_TIME = time.time()

# ---------------- User config ----------------
RUN_TYPE          = "Original Only"
SEED              = 42
SAMPLE_SIZE       = 10000
TEST_RATIO        = 0.20
MODEL_ID          = "microsoft/Phi-4-mini-instruct"
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
USE_4BIT          = torch.cuda.is_available()

# Training settings: checkpoint-1000 غالبا كمل epoch 1، وده هيكمل لحد 2 epochs
NUM_EPOCHS        = 2
TRAIN_BATCH_SIZE  = 1
GRAD_ACCUM        = 8
LEARNING_RATE     = 1e-4
MAX_SEQ_LENGTH    = 512
WARMUP_RATIO      = 0.05
LR_SCHEDULER      = "cosine"
OPTIM             = "paged_adamw_8bit"

LORA_R            = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.05

MAX_NEW_TOKENS    = 32
INF_BATCH_SIZE    = 1
SEM_MODEL_ID      = "sentence-transformers/all-MiniLM-L6-v2"

INPUT_DIR   = Path("/kaggle/input")
WORK_DIR    = Path("/kaggle/working")
OUTPUT_DIR  = WORK_DIR / "PHI4_ORIGINAL_ONLY_RAG"
MODELS_DIR  = OUTPUT_DIR / "models"
MODEL_DIR   = MODELS_DIR / "phi4mini_original_only_lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("/kaggle/input/datasets/belalheshamfathy/cvbhjkuyg/qa_data.json"),
    Path("/kaggle/input/qwertyu/qa_data.json"),
]

def fmt_time(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def gpu_report(prefix="GPU"):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved  = torch.cuda.memory_reserved() / 1024**3
        name = torch.cuda.get_device_name(0)
        print(f"{prefix}: {name} | allocated={allocated:.2f}GB | reserved={reserved:.2f}GB")
    else:
        print(f"{prefix}: CPU mode")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print("Visible CUDA GPUs:", torch.cuda.device_count())
    gpu_report("Initial GPU")

print("Run type:", RUN_TYPE)
print("Device:", DEVICE)
print("Model:", MODEL_ID)
print("Output dir:", OUTPUT_DIR)
print("Target model dir:", MODEL_DIR)

Visible CUDA GPUs: 2
Initial GPU: Tesla T4 | allocated=0.00GB | reserved=0.00GB
Run type: Original Only
Device: cuda
Model: microsoft/Phi-4-mini-instruct
Output dir: /kaggle/working/PHI4_ORIGINAL_ONLY_RAG
Target model dir: /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora


In [6]:
# ============================================================
# CELL 3 — Locate data + load original dataset
# ============================================================
def load_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            return pd.DataFrame([data])
        raise ValueError("Unsupported JSON format")
    except json.JSONDecodeError:
        return pd.read_json(path, lines=True)

DATA_PATH = None
for p in DATA_CANDIDATES:
    if p.exists():
        DATA_PATH = p
        break
if DATA_PATH is None:
    found = list(INPUT_DIR.rglob("qa_data.json"))
    if not found:
        raise FileNotFoundError("qa_data.json not found under /kaggle/input. Add the original QA dataset first.")
    DATA_PATH = found[0]

print("Using DATA_PATH:", DATA_PATH)
_df = load_json(DATA_PATH)
print(f"Loaded: {len(_df):,} rows | columns: {list(_df.columns)}")

required_cols = ["question", "right_answer", "knowledge"]
missing = [c for c in required_cols if c not in _df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

_df["_context"] = _df["knowledge"].fillna("").astype(str)
_df["question"] = _df["question"].fillna("").astype(str)
_df["right_answer"] = _df["right_answer"].fillna("").astype(str)
_df = _df[
    (_df["_context"].str.strip() != "") &
    (_df["question"].str.strip() != "") &
    (_df["right_answer"].str.strip() != "")
].reset_index(drop=True)
print(f"Usable rows after cleaning: {len(_df):,}")

Using DATA_PATH: /kaggle/input/datasets/belalheshamfathy/cvbhjkuyg/qa_data.json
Loaded: 10,000 rows | columns: ['knowledge', 'question', 'right_answer', 'hallucinated_answer']
Usable rows after cleaning: 10,000


In [7]:
# ============================================================
# CELL 4 — Manual train/test split without sklearn
# ============================================================
if SAMPLE_SIZE and len(_df) > SAMPLE_SIZE:
    idx = np.random.choice(len(_df), SAMPLE_SIZE, replace=False)
    idx.sort()
    _df = _df.iloc[idx].reset_index(drop=True)
    print(f"Sub-sampled to {SAMPLE_SIZE:,} records.")

rng = np.random.default_rng(SEED)
indices = np.arange(len(_df))
rng.shuffle(indices)
test_n = int(round(len(_df) * TEST_RATIO))
test_idx = indices[:test_n]
train_idx = indices[test_n:]
train_df = _df.iloc[train_idx].reset_index(drop=True)
test_df  = _df.iloc[test_idx].reset_index(drop=True)
print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")

Train: 8,000 | Test: 2,000


In [8]:
# ============================================================
# CELL 5 — Prompt + dataset helpers
# ============================================================
ORIG_SYSTEM = """You are an accurate question-answering assistant.
Answer the question based solely on the provided context.
If the answer is not in the context, reply: I don't know.
Be concise."""

SYSTEM_PROMPT = ORIG_SYSTEM

def build_input_prompt(system: str, context: str, question: str) -> str:
    return (f"<|system|>\n{system}\n<|end|>\n"
            f"<|user|>\nContext:\n{context.strip()}\n\nQuestion: {question.strip()}\n<|end|>\n"
            f"<|assistant|>\n")

def build_full_example(system: str, context: str, question: str, answer: str) -> str:
    return build_input_prompt(system, context, question) + answer.strip() + "<|end|>"

def make_hf_dataset(df: pd.DataFrame, system: str, context_col: str = "_context") -> Dataset:
    examples = []
    for _, row in df.iterrows():
        ctx = str(row[context_col] or "").strip()
        q   = str(row["question"] or "").strip()
        a   = str(row["right_answer"] or "").strip()
        if ctx and q and a:
            examples.append({"text": build_full_example(system, ctx, q, a)})
    return Dataset.from_list(examples)

print("Prompt helpers ready.")

Prompt helpers ready.


In [9]:
# ============================================================
# CELL 6 — Training helpers
# ============================================================
def single_gpu_device_map():
    return {"": 0} if torch.cuda.is_available() else None

def get_compute_dtype():
    return torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

def build_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=get_compute_dtype(),
        bnb_4bit_use_double_quant=True,
    )

def load_base_model_for_training():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    kwargs = dict(trust_remote_code=False, low_cpu_mem_usage=True, attn_implementation="sdpa", torch_dtype=get_compute_dtype())
    if USE_4BIT:
        kwargs.update(quantization_config=build_bnb_config(), device_map=single_gpu_device_map())
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
    model.config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    return model, tokenizer

def discover_lora_targets(model):
    preferred = ["qkv_proj", "q_proj", "k_proj", "v_proj", "o_proj", "gate_up_proj", "gate_proj", "up_proj", "down_proj"]
    names = set()
    for name, module in model.named_modules():
        last = name.split(".")[-1]
        if last in preferred:
            names.add(last)
    targets = [x for x in preferred if x in names]
    if not targets:
        targets = ["q_proj", "k_proj", "v_proj", "o_proj"]
    print("LoRA target modules:", targets)
    return targets

def attach_lora(model):
    if USE_4BIT:
        model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=discover_lora_targets(model),
    )
    return get_peft_model(model, lora_config)

class RuntimeCallback(TrainerCallback):
    def __init__(self, label):
        self.label = label
        self.start = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.start = time.time()
        print("\n" + "="*80)
        print(f"START TRAINING: {self.label}")
        print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        gpu_report("GPU at start")
        print("="*80)
    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        if "loss" in logs:
            elapsed = fmt_time(time.time() - self.start) if self.start else "00:00:00"
            lr = logs.get("learning_rate", None)
            msg = f"[{self.label}] step {state.global_step:>5} | elapsed {elapsed} | loss {logs['loss']:.4f}"
            if lr is not None:
                msg += f" | lr {lr:.2e}"
            print(msg)
    def on_train_end(self, args, state, control, **kwargs):
        elapsed = fmt_time(time.time() - self.start) if self.start else "00:00:00"
        print("\n" + "="*80)
        print(f"END TRAINING: {self.label} | total time {elapsed}")
        gpu_report("GPU at end")
        print("="*80)

def get_training_args(output_dir, run_name):
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    return TrainingArguments(
        output_dir=output_dir,
        run_name=f"phi4_lora_{run_name}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=LR_SCHEDULER,
        optim=OPTIM,
        logging_steps=5,
        save_strategy="epoch",
        save_total_limit=3,
        fp16=torch.cuda.is_available() and not use_bf16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        report_to="none",
        seed=SEED,
        remove_unused_columns=True,
    )

def tokenize_sft_dataset(dataset, tokenizer):
    def _tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)
    return dataset.map(_tok, batched=True, remove_columns=dataset.column_names)

def build_sft_trainer(model, args, train_dataset, tokenizer, label):
    tokenized_train = tokenize_sft_dataset(train_dataset, tokenizer)
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    return Trainer(model=model, args=args, train_dataset=tokenized_train, data_collator=collator, callbacks=[RuntimeCallback(label)])

print("Training helpers ready.")

Training helpers ready.


In [10]:
# ============================================================
# CELL 7 — Locate/copy last checkpoint then resume training
# ============================================================
def checkpoint_step(path: Path):
    m = re.search(r"checkpoint-(\d+)$", path.name)
    return int(m.group(1)) if m else -1

def find_checkpoints(root: Path):
    if not root.exists():
        return []
    ckpts = []
    for p in root.rglob("checkpoint-*"):
        if p.is_dir() and (p / "trainer_state.json").exists():
            ckpts.append(p)
    return sorted(ckpts, key=checkpoint_step)

def copy_previous_output_to_working():
    """لو الموديل موجود كـ Kaggle Dataset في /kaggle/input/phi4_ooorg، انسخه للـ working عشان نكمل تدريب."""
    if (MODEL_DIR / "adapter_config.json").exists() or find_checkpoints(MODEL_DIR):
        print("Existing working model/checkpoint found:", MODEL_DIR)
        return

    candidates = []
    for p in INPUT_DIR.rglob("phi4mini_original_only_lora"):
        if p.is_dir() and ((p / "adapter_config.json").exists() or find_checkpoints(p)):
            candidates.append(p)
    if not candidates:
        print("No previous original LoRA folder found in /kaggle/input. Training will start from scratch.")
        return

    src = candidates[0]
    print("Copying previous LoRA/checkpoint from:", src)
    print("to:", MODEL_DIR)
    if MODEL_DIR.exists():
        shutil.rmtree(MODEL_DIR)
    shutil.copytree(src, MODEL_DIR)

copy_previous_output_to_working()
LAST_CHECKPOINT = None
ckpts = find_checkpoints(MODEL_DIR)
if ckpts:
    LAST_CHECKPOINT = ckpts[-1]
    print("Will resume from checkpoint:", LAST_CHECKPOINT)
else:
    print("No trainer checkpoint found. Will train from scratch.")

print("Files in MODEL_DIR:")
for p in sorted(MODEL_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(MODEL_DIR))

Existing working model/checkpoint found: /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora
Will resume from checkpoint: /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora/checkpoint-1000
Files in MODEL_DIR:
- checkpoint-1000/README.md
- checkpoint-1000/adapter_config.json
- checkpoint-1000/adapter_model.safetensors
- checkpoint-1000/optimizer.pt
- checkpoint-1000/rng_state.pth
- checkpoint-1000/scheduler.pt
- checkpoint-1000/trainer_state.json
- checkpoint-1000/training_args.bin


In [11]:
# ============================================================
# CELL 8 — Resume/finish training
# ============================================================
print("=" * 60)
print(f"TRAINING/RESUME: Phi-4-mini + {RUN_TYPE} Knowledge LoRA")
print("=" * 60)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

hf_train = make_hf_dataset(train_df, SYSTEM_PROMPT, "_context")
print(f"Training examples: {len(hf_train):,}")
print("Example:")
print(hf_train[0]["text"][:400] + "...")

model_train, tokenizer_train = load_base_model_for_training()
model_train = attach_lora(model_train)

trainable_params = sum(p.numel() for p in model_train.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_train.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,} ({trainable_params/total_params:.2%})")

train_args = get_training_args(str(MODEL_DIR), RUN_TYPE.lower().replace(" ", "_"))
trainer = build_sft_trainer(model_train, train_args, hf_train, tokenizer_train, f"Phi-4 {RUN_TYPE} LoRA")

t_train_start = time.time()
if LAST_CHECKPOINT is not None:
    print("Resuming trainer from:", LAST_CHECKPOINT)
    trainer.train(resume_from_checkpoint=str(LAST_CHECKPOINT))
else:
    print("Starting training from scratch.")
    trainer.train()
train_time = time.time() - t_train_start

trainer.save_model(str(MODEL_DIR))
tokenizer_train.save_pretrained(str(MODEL_DIR))
print(f"Saved final LoRA adapter to: {MODEL_DIR}")
print(f"Training time this run: {fmt_time(train_time)}")

del trainer, model_train, tokenizer_train
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TRAINING/RESUME: Phi-4-mini + Original Only Knowledge LoRA


This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Training examples: 8,000
Example:
<|system|>
You are an accurate question-answering assistant.
Answer the question based solely on the provided context.
If the answer is not in the context, reply: I don't know.
Be concise.
<|end|>
<|user|>
Context:
How to Make Love to a Woman is a 2010 American comedy film directed by Scott Culver and written by Dennis Kao, both making their respective debuts, starring Josh Meyers, Krysten Ritter,...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Resuming trainer from: /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora/checkpoint-1000


	per_device_train_batch_size: 1 (from args) != 0 (from trainer_state.json)



START TRAINING: Phi-4 Original Only LoRA
Time: 2026-06-12 16:08:58
GPU at start: Tesla T4 | allocated=3.97GB | reserved=6.30GB


Step,Training Loss
1005,2.361560
1010,2.351981
1015,2.300449
1020,2.025363
1025,2.223005
1030,2.422904
1035,2.332320
1040,2.233449
1045,2.461309
1050,2.262930


[Phi-4 Original Only LoRA] step  1005 | elapsed 00:01:23 | loss 2.3616 | lr 5.38e-05
[Phi-4 Original Only LoRA] step  1010 | elapsed 00:02:46 | loss 2.3520 | lr 5.34e-05
[Phi-4 Original Only LoRA] step  1015 | elapsed 00:04:08 | loss 2.3004 | lr 5.30e-05
[Phi-4 Original Only LoRA] step  1020 | elapsed 00:05:31 | loss 2.0254 | lr 5.26e-05
[Phi-4 Original Only LoRA] step  1025 | elapsed 00:06:59 | loss 2.2230 | lr 5.21e-05
[Phi-4 Original Only LoRA] step  1030 | elapsed 00:08:24 | loss 2.4229 | lr 5.17e-05
[Phi-4 Original Only LoRA] step  1035 | elapsed 00:09:49 | loss 2.3323 | lr 5.13e-05
[Phi-4 Original Only LoRA] step  1040 | elapsed 00:11:14 | loss 2.2334 | lr 5.09e-05
[Phi-4 Original Only LoRA] step  1045 | elapsed 00:12:45 | loss 2.4613 | lr 5.05e-05
[Phi-4 Original Only LoRA] step  1050 | elapsed 00:14:07 | loss 2.2629 | lr 5.01e-05
[Phi-4 Original Only LoRA] step  1055 | elapsed 00:15:35 | loss 2.4849 | lr 4.97e-05
[Phi-4 Original Only LoRA] step  1060 | elapsed 00:17:01 | loss 2

In [12]:
# ============================================================
# CELL 9 — Zip final model adapter
# ============================================================
def create_zip(folder: Path, zip_path: Path):
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fpath in folder.rglob("*"):
            if fpath.is_file():
                zf.write(fpath, fpath.relative_to(folder.parent))
    size_mb = zip_path.stat().st_size / 1024 / 1024
    print(f"Created: {zip_path} ({size_mb:.1f} MB)")

zip_path = OUTPUT_DIR / f"{MODEL_DIR.name}.zip"
create_zip(MODEL_DIR, zip_path)

Created: /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/phi4mini_original_only_lora.zip (330.6 MB)


In [13]:
# ============================================================
# CELL 10 — Load final adapter for inference
# ============================================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading base model for inference:", MODEL_ID)
bnb_inf = build_bnb_config() if USE_4BIT else None
base_kwargs = dict(
    trust_remote_code=False,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    torch_dtype=get_compute_dtype(),
)
if USE_4BIT:
    base_kwargs.update(quantization_config=bnb_inf, device_map=single_gpu_device_map())

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **base_kwargs)
base_model.config.use_cache = True
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False, use_fast=True)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "left"

model_inf = PeftModel.from_pretrained(base_model, str(MODEL_DIR))
print("Inference model loaded from:", MODEL_DIR)
gpu_report("GPU after inference load")

Loading base model for inference: microsoft/Phi-4-mini-instruct


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Inference model loaded from: /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora
GPU after inference load: Tesla T4 | allocated=5.09GB | reserved=9.58GB


In [14]:
# ============================================================
# CELL 11 — Evaluation metric helpers (NO sklearn)
# ============================================================
print(f"Loading semantic model: {SEM_MODEL_ID}")
sem_model = SentenceTransformer(SEM_MODEL_ID)
print("Semantic model loaded.")

def normalize_answer(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    stopwords = {"the","a","an","is","are","was","were","of","in","on","at","to","for","with","by"}
    return " ".join([t for t in s.split() if t not in stopwords])

def exact_match(p, g):
    return float(normalize_answer(p) == normalize_answer(g))

def token_f1(p, g):
    pt = normalize_answer(p).split()
    gt = normalize_answer(g).split()
    if len(pt) == 0 or len(gt) == 0:
        return float(pt == gt)
    com = Counter(pt) & Counter(gt)
    nc = sum(com.values())
    if nc == 0:
        return 0.0
    prec = nc / len(pt)
    rec = nc / len(gt)
    return 2 * prec * rec / (prec + rec)

def fuzzy_score(p, g):
    return rfuzz.token_sort_ratio(str(p).lower(), str(g).lower()) / 100.0

def cosine_np(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def sem_sim_batch(preds, golds):
    ep = sem_model.encode(preds, batch_size=64, show_progress_bar=False)
    eg = sem_model.encode(golds, batch_size=64, show_progress_bar=False)
    return [cosine_np(a, b) for a, b in zip(ep, eg)]

def answer_in_ctx(p, c):
    return float(rfuzz.partial_ratio(normalize_answer(p), normalize_answer(c)) >= 50)

def halluc_flag(p, c, g):
    if exact_match(p, g) or token_f1(p, g) > 0.4 or fuzzy_score(p, g) > 0.5:
        return 0.0
    return 0.0 if rfuzz.partial_ratio(normalize_answer(p), normalize_answer(c)) >= 50 else 1.0

def ctx_util(p, c):
    pt = set(normalize_answer(p).split())
    ct = set(normalize_answer(c).split())
    return len(pt & ct) / max(len(pt), 1)

def evaluate_all(preds, golds, contexts):
    em  = [exact_match(p,g) for p,g in zip(preds,golds)]
    f1  = [token_f1(p,g)    for p,g in zip(preds,golds)]
    fuz = [fuzzy_score(p,g) for p,g in zip(preds,golds)]
    sem = sem_sim_batch(preds, golds)
    aic = [answer_in_ctx(p,c) for p,c in zip(preds,contexts)]
    hal = [halluc_flag(p,c,g) for p,c,g in zip(preds,contexts,golds)]
    cut = [ctx_util(p,c) for p,c in zip(preds,contexts)]
    agg = [(a+b+c)/3 for a,b,c in zip(f1,sem,fuz)]
    return {
        "exact_match": round(float(np.mean(em)),4),
        "accuracy_threshold_50": round(float(np.mean([float(s>=0.50) for s in f1])),4),
        "accuracy_threshold_70": round(float(np.mean([float(s>=0.70) for s in f1])),4),
        "mean_token_f1": round(float(np.mean(f1)),4),
        "mean_fuzzy": round(float(np.mean(fuz)),4),
        "mean_semantic_similarity": round(float(np.mean(sem)),4),
        "mean_aggregate_score": round(float(np.mean(agg)),4),
        "answer_in_context_rate": round(float(np.mean(aic)),4),
        "hallucination_rate": round(float(np.mean(hal)),4),
        "context_utilization": round(float(np.mean(cut)),4),
        "_em":em,"_f1":f1,"_fuz":fuz,"_sem":sem,"_agg":agg,"_aic":aic,"_hal":hal,"_cut":cut,
    }
print("Evaluation helpers ready.")

Loading semantic model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic model loaded.
Evaluation helpers ready.


In [15]:
# ============================================================
# CELL 12 — Inference
# ============================================================
GEN_PARAMS = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    temperature=1.0,
    repetition_penalty=1.05,
    pad_token_id=base_tokenizer.pad_token_id,
    eos_token_id=base_tokenizer.eos_token_id,
)

def extract_answer(full_text, prompt):
    if full_text.startswith(prompt):
        ans = full_text[len(prompt):]
    elif "<|assistant|>" in full_text:
        ans = full_text.split("<|assistant|>")[-1]
    else:
        ans = full_text
    stop_markers = ["<|user|>", "<|system|>", "<|assistant|>", "<|end|>", "<|endoftext|>", "\nContext:", "\nQuestion:", "Context:", "Question:"]
    for marker in stop_markers:
        if marker in ans:
            ans = ans.split(marker)[0]
    ans = re.sub(r"^(answer\s*:\s*)", "", ans, flags=re.I).strip()
    ans = ans.strip(" \n\t:.-")
    lines = [l.strip() for l in ans.split("\n") if l.strip()]
    return lines[0] if lines else "I don't know"

def run_inference(peft_model, tokenizer, df_test, system_prompt, context_col="_context", desc="Generating"):
    peft_model.eval()
    predictions = []
    records = df_test.to_dict("records")
    batches = [records[i:i+INF_BATCH_SIZE] for i in range(0, len(records), INF_BATCH_SIZE)]
    start = time.time()
    for batch in tqdm(batches, desc=desc):
        prompts = []
        for row in batch:
            ctx = str(row.get(context_col, "") or "").strip()
            q = str(row.get("question", "") or "").strip()
            prompts.append(build_input_prompt(system_prompt, ctx, q))
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(peft_model.device)
        with torch.no_grad():
            out = peft_model.generate(**enc, **GEN_PARAMS)
        for i, ids in enumerate(out):
            full = tokenizer.decode(ids, skip_special_tokens=False)
            predictions.append(extract_answer(full, prompts[i]))
    print(f"{desc} finished in {fmt_time(time.time() - start)}")
    return predictions

print("=" * 60)
print(f"INFERENCE: Fine-Tuned Phi-4-mini + {RUN_TYPE} LoRA")
print("=" * 60)
t0 = time.time()
preds = run_inference(model_inf, base_tokenizer, test_df, SYSTEM_PROMPT, "_context", desc=f"{RUN_TYPE} LoRA Inference")
inference_time = time.time() - t0
print(f"Inference time: {fmt_time(inference_time)}")

for i in range(min(3, len(preds))):
    print(f"\n--- Example {i+1} ---")
    print(f"Q   : {test_df['question'].iloc[i]}")
    print(f"Ref : {test_df['right_answer'].iloc[i]}")
    print(f"Pred: {preds[i]}")

INFERENCE: Fine-Tuned Phi-4-mini + Original Only LoRA


Original Only LoRA Inference:   0%|          | 0/2000 [00:00<?, ?it/s]

Original Only LoRA Inference finished in 00:51:47
Inference time: 00:51:47

--- Example 1 ---
Q   : Simon WIlliam Gamble played non league football for what club located in Lincolnshire, England?
Ref : Grantham Town Football Club
Pred: Grantham Town Football Club

--- Example 2 ---
Q   : Both Janggi and Isola are board-based strategy games; however, which game is played on a 9x10 gameboard similar to Chinese Chess?
Ref : Janggi
Pred: Janggi

--- Example 3 ---
Q   : Where was the film A Taxi Driver released?
Ref : South Korea
Pred: South Korea


In [16]:
# ============================================================
# CELL 13 — Evaluate + save outputs
# ============================================================
golds = list(test_df["right_answer"])
contexts = list(test_df["_context"])

print(f"Evaluating {RUN_TYPE} model...")
eval_start = time.time()
metrics = evaluate_all(preds, golds, contexts)
print(f"Evaluation time: {fmt_time(time.time() - eval_start)}")

DISPLAY_METRIC_MAP = {
    "accuracy_by_threshold": "accuracy_threshold_50",
    "exact_match": "exact_match",
    "mean_token_f1": "mean_token_f1",
    "mean_fuzzy": "mean_fuzzy",
    "mean_aggregate": "mean_aggregate_score",
    "answer_in_context": "answer_in_context_rate",
    "hallucination_rate": "hallucination_rate",
    "mean_semantic_similarity": "mean_semantic_similarity",
}
metric_rows = [{"metric": shown_name, "value": round(metrics[metric_key], 6)} for shown_name, metric_key in DISPLAY_METRIC_MAP.items()]
df_metrics = pd.DataFrame(metric_rows)
print(df_metrics.to_string(index=False))

pred_df = test_df[["question", "right_answer"]].copy()
pred_df["predicted_answer"] = preds
pred_df["exact_match"]       = metrics["_em"]
pred_df["token_f1"]          = metrics["_f1"]
pred_df["fuzzy_score"]       = metrics["_fuz"]
pred_df["semantic_sim"]      = metrics["_sem"]
pred_df["aggregate_score"]   = metrics["_agg"]
pred_df["answer_in_context"] = metrics["_aic"]
pred_df["hallucinated"]      = metrics["_hal"]
pred_df["context_util"]      = metrics["_cut"]
pred_df["method"]            = "original_only"

predictions_csv = OUTPUT_DIR / "original_only_predictions.csv"
metrics_csv = OUTPUT_DIR / "original_only_metrics.csv"
summary_txt = OUTPUT_DIR / "original_only_summary.txt"
pred_df.to_csv(predictions_csv, index=False)
df_metrics.to_csv(metrics_csv, index=False)

summary = f"""
========================================================
FINE-TUNED PHI-4-MINI RAG - ORIGINAL ONLY
========================================================

Dataset     : {len(test_df)} test examples
Model       : {MODEL_ID}
LoRA rank   : {LORA_R} | Alpha: {LORA_ALPHA} | Dropout: {LORA_DROPOUT}
Epochs      : {NUM_EPOCHS} | LR: {LEARNING_RATE}
Seed        : {SEED}
Train time this run: {fmt_time(train_time)}
Infer time  : {fmt_time(inference_time)}

--- METRICS ---
{df_metrics.to_string(index=False)}

--- SAVED FILES ---
Adapter folder : {MODEL_DIR}
Adapter zip    : {zip_path}
Predictions CSV: {predictions_csv}
Metrics CSV    : {metrics_csv}
""".strip()
print(summary)
summary_txt.write_text(summary, encoding="utf-8")
print(f"Summary saved to: {summary_txt}")

Evaluating Original Only model...
Evaluation time: 00:00:00
                  metric  value
   accuracy_by_threshold 0.9130
             exact_match 0.8280
           mean_token_f1 0.8911
              mean_fuzzy 0.9018
          mean_aggregate 0.9073
       answer_in_context 0.9965
      hallucination_rate 0.0025
mean_semantic_similarity 0.9291
FINE-TUNED PHI-4-MINI RAG - ORIGINAL ONLY

Dataset     : 2000 test examples
Model       : microsoft/Phi-4-mini-instruct
LoRA rank   : 16 | Alpha: 32 | Dropout: 0.05
Epochs      : 2 | LR: 0.0001
Seed        : 42
Train time this run: 04:48:08
Infer time  : 00:51:47

--- METRICS ---
                  metric  value
   accuracy_by_threshold 0.9130
             exact_match 0.8280
           mean_token_f1 0.8911
              mean_fuzzy 0.9018
          mean_aggregate 0.9073
       answer_in_context 0.9965
      hallucination_rate 0.0025
mean_semantic_similarity 0.9291

--- SAVED FILES ---
Adapter folder : /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models

In [17]:
# ============================================================
# CELL 14 — Final file list
# ============================================================
def folder_size_mb(path):
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return round(total / 1024 / 1024, 2)

def zip_size_mb(path):
    return round(Path(path).stat().st_size / 1024 / 1024, 2) if Path(path).exists() else 0

print("\n" + "="*70)
print("MODEL SAVE LOCATION")
print("="*70)
print(f"LoRA folder     : {MODEL_DIR}")
print(f"Folder size     : {folder_size_mb(MODEL_DIR):.2f} MB")
print(f"Zip file        : {zip_path}")
print(f"Zip size        : {zip_size_mb(zip_path):.2f} MB")
print(f"Training time   : {fmt_time(train_time)}")
print(f"Inference time  : {fmt_time(inference_time)}")
print(f"Trainable params: {trainable_params:,}")

print("\nSAVED OUTPUT FILES:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print("-", f)


MODEL SAVE LOCATION
LoRA folder     : /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora
Folder size     : 384.46 MB
Zip file        : /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/phi4mini_original_only_lora.zip
Zip size        : 330.58 MB
Training time   : 04:48:08
Inference time  : 00:51:47
Trainable params: 23,068,672

SAVED OUTPUT FILES:
- /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora/README.md
- /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora/adapter_config.json
- /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora/adapter_model.safetensors
- /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora/chat_template.jinja
- /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora/checkpoint-1000/README.md
- /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/models/phi4mini_original_only_lora/checkpoint-1000/adapter_config.json
- /kaggle/working/PHI4_ORIGINAL_ONLY_RAG/mod